In [1]:
import pandas as pd
import numpy as np

In [2]:
rng = np.random.default_rng(42)   # fixed seed so results are reproducible
n = 200                           # number of rows

df = pd.DataFrame({
    'protein_consumed_per_week': rng.integers(500, 1500, n),   # grams
    'carbs_consumed_per_week':   rng.integers(1500, 4000, n),  # grams
    'fats_consumed_per_week':    rng.integers(300, 1000, n),   # grams
    'fibres_consumed_per_week':  rng.integers(100, 400, n),    # grams
})

# Bulking more likely with high protein + carbs, plus random noise
score = (df['protein_consumed_per_week'] / 1500
         + df['carbs_consumed_per_week'] / 4000
         + rng.normal(0, 0.3, n))
df['bulked'] = (score > score.median()).astype(int)

In [3]:
df.head(10)

,protein_consumed_per_week,carbs_consumed_per_week,fats_consumed_per_week,fibres_consumed_per_week,bulked
0,589,2415,998,284,0
1,1273,3771,844,333,1
2,1154,2737,524,321,1
3,938,3249,980,140,0
4,933,2643,649,347,0
5,1358,2164,650,260,1
6,585,3410,603,196,0
7,1197,3922,400,254,1
8,701,2158,954,390,0
9,594,3446,309,357,0


In [4]:
X = df.drop(columns=['bulked'])
y = df['bulked']

In [5]:
from sklearn.model_selection import train_test_split

In [7]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=28, stratify=y)

In [9]:
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression

In [10]:
scaler = StandardScaler()
model = LogisticRegression()
pipe = make_pipeline(scaler, model)

In [11]:
from sklearn.model_selection import GridSearchCV, cross_val_score

In [12]:
scores = cross_val_score(pipe, X_train, y_train, cv=5, scoring='accuracy')

In [13]:
print(scores)

[0.625   0.78125 0.59375 0.65625 0.65625]


In [14]:
print('Mean accuracy:', scores.mean())

Mean accuracy: 0.6625


In [17]:
param_grid = {
    'logisticregression__C': [0.01, 0.1, 1, 10, 100],
    'logisticregression__penalty': ['l2'],
    'logisticregression__solver': ['lbfgs']
}

In [18]:
grid = GridSearchCV(pipe, param_grid, cv=5, scoring='accuracy')
grid.fit(X_train, y_train)

GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('standardscaler', StandardScaler()),
                                       ('logisticregression',
                                        LogisticRegression())]),
             param_grid={'logisticregression__C': [0.01, 0.1, 1, 10, 100],
                         'logisticregression__penalty': ['l2'],
                         'logisticregression__solver': ['lbfgs']},
             scoring='accuracy')

In [19]:
print('Best params:', grid.best_params_)
print('Best CV score:', grid.best_score_)

Best params: {'logisticregression__C': 0.01, 'logisticregression__penalty': 'l2', 'logisticregression__solver': 'lbfgs'}
Best CV score: 0.6625


In [21]:
from sklearn.metrics import classification_report

In [22]:
best_model = grid.best_estimator_
y_pred = best_model.predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.79      0.75      0.77        20
           1       0.76      0.80      0.78        20

    accuracy                           0.78        40
   macro avg       0.78      0.78      0.77        40
weighted avg       0.78      0.78      0.77        40

